# 05b - (Tahap 3) Cleaning dataset with RAGAS

In [1]:
!pip install "ragas==0.2.*" langchain-openai langchain-community langchain-huggingface deepeval -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.9/190.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 429.6/429.6 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 1.

In [2]:
import torch
import sys
import csv
import shutil
import platform
from types import ModuleType
from collections import defaultdict
 
_mock_vertex = ModuleType("langchain_community.chat_models.vertexai")
_mock_vertex.ChatVertexAI = None
sys.modules["langchain_community.chat_models.vertexai"] = _mock_vertex
 
import os
os.environ["RAGAS_DO_NOT_TRACK"] = "true"
os.environ.setdefault("DEEPEVAL_MAX_RETRIES", "1")
 
import pandas as pd
import numpy as np
import json
import math
import random
import time
import datetime
from pathlib import Path

## 1. Konfigurasi

In [3]:
def set_global_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    
OUTPUT_05B_DIR = "/kaggle/input/notebooks/muhammadnabilul/05b-tahap-2-cleaning-dataset-with-rule-based/output_05b"
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device={DEVICE} cuda={torch.cuda.is_available()} "
      f"name={torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'}")
 
CONFIG = {
    "input_path": f"{OUTPUT_05B_DIR}/filtered_records.jsonl",
    "output_dir": "/kaggle/working/qc",

    # Embed config
    "embedding_model": "firqaaa/Indo-Sentence-BERT-Base",
    "embed_batch_size": 32,
    "indobert_threshold_flag": 0.5,

    # Stratified config
    "min_per_stratum": 30,
    "max_per_stratum": None,
    "flag_ratio": 0.5,
    "stratify_key": "subtask_type",

    # Ragas config
    "ragas_judge_model": "gpt-4o-mini",
    "ragas_relevancy_strictness": 1,
    "ragas_embedding_model": "nomic-ai/nomic-embed-text-v1.5",
    "ragas_max_workers": 4,
    "ragas_max_retries": 5,
    "request_delay": 1.0,
    "ragas_sample_rate": 0.10,
    
    # Deepeval config
    "deepeval_metrics": ["faithfulness", "answer_relevancy"],
    "deepeval_judge_model": "gpt-4o-mini",
    "deepeval_request_timeout": 60,
    "deepeval_max_retries": 2,
    "deepeval_request_delay": 1.0,
    "API_secret_name": "API_OpenAI",

    # Human eval config
    "human_eval_n": 100,
    "human_borderline_frac": 0.6,

    "seed": 42,
    "device": DEVICE,

    # Test Mode
    "test_mode": False,
    "max_strata_test": 2,
    "max_samples_per_stratum_test": 3,
    "human_eval_n_test": 4,
}
 
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
set_global_seed(CONFIG["seed"])

device=cpu cuda=False name=-


### Log ENV

In [4]:
def log_environment(config):
    import importlib.metadata as md
 
    packages = ["ragas", "deepeval", "openai", "langchain-openai",
                "langchain-huggingface", "sentence-transformers", "torch"]
    versions = {}
    for p in packages:
        try:
            versions[p] = md.version(p)
        except Exception:
            versions[p] = "not installed"
 
    env = {
        "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "python": platform.python_version(),
        "platform": platform.platform(),
        "device": config["device"],
        "cuda_available": torch.cuda.is_available(),
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "seed": config["seed"],
        "packages": versions,
        "judge_models": {
            "ragas": config["ragas_judge_model"],
            "deepeval": config["deepeval_judge_model"],
            "deepeval_metrics": config.get("deepeval_metrics"),
        },
        "sampling": {
            "ragas_sample_rate": config["ragas_sample_rate"],
            "min_per_stratum": config["min_per_stratum"],
            "flag_ratio": config["flag_ratio"],
        },
    }
 
    path = f"{config['output_dir']}/environment_snapshot.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(env, f, ensure_ascii=False, indent=2)
 
    print("environment snapshot:")
    for k, v in versions.items():
        print(f"  {k}: {v}")
    print(f"disimpan ke: {path}")
    return env
 
 
env_snapshot = log_environment(CONFIG)

print(env_snapshot["judge_models"])

environment snapshot:
  ragas: 0.2.15
  deepeval: 4.1.3
  openai: 2.48.0
  langchain-openai: 1.4.1
  langchain-huggingface: 1.2.2
  sentence-transformers: 5.4.1
  torch: 2.10.0+cpu
disimpan ke: /kaggle/working/qc/environment_snapshot.json
{'ragas': 'gpt-4o-mini', 'deepeval': 'gpt-4o-mini', 'deepeval_metrics': ['faithfulness', 'answer_relevancy']}


### Resume safe

In [5]:
def restore_checkpoints_from_previous_version(config, dataset_slug=None):
    """
    Menyalin checkpoint dari output versi notebook Kaggle sebelumnya
    (yang muncul read-only di /kaggle/input) ke working directory,
    agar fungsi resume di tiap stage bisa membacanya dan melanjutkan.
    """
    working_dir = Path(config["output_dir"])
    working_dir.mkdir(parents=True, exist_ok=True)
 
    files_to_restore = [
        "stage1_prefilter.csv",
        "stage1_summary.json",
        "stage2_ragas.csv",
        "stage2_ragas_checkpoint.csv",
        "stage2c_deepeval_checkpoint.csv",
        "stage2_ragas_test.csv",
        "stage2_ragas_checkpoint_test.csv",
        "stage2c_deepeval_checkpoint_test.csv",
    ]
 
    kaggle_input = Path("/kaggle/input")
 
    restored = []
    for fname in files_to_restore:
        target = working_dir / fname
        if target.exists() and target.stat().st_size > 0:
            continue
        if not kaggle_input.exists():
            break
        candidates = [p for p in kaggle_input.rglob(fname)
                      if p.is_file() and p.stat().st_size > 0]
        if not candidates:
            continue
        source = max(candidates, key=lambda p: p.stat().st_mtime)
        shutil.copy2(source, target)
        restored.append(f"{fname} <- {source.parent}")
 
    if restored:
        print("checkpoint dipulihkan dari versi sebelumnya:")
        for r in restored:
            print(f"  {r}")
    else:
        print("tidak ada checkpoint lama ditemukan (atau file sudah ada di working dir).")
 
    print("\nisi working dir saat ini:")
    for f in sorted(working_dir.iterdir()):
        print(f"  {f.name}  {f.stat().st_size} bytes")
 
    return restored
 
 
restore_checkpoints_from_previous_version(CONFIG)

checkpoint dipulihkan dari versi sebelumnya:
  stage1_prefilter.csv <- /kaggle/input/datasets/muhammadnabilul/checkpoint-05b-tahap-3/qc
  stage1_summary.json <- /kaggle/input/datasets/muhammadnabilul/checkpoint-05b-tahap-3/qc
  stage2_ragas.csv <- /kaggle/input/datasets/muhammadnabilul/checkpoint-05b-tahap-3/qc
  stage2_ragas_checkpoint.csv <- /kaggle/input/datasets/muhammadnabilul/checkpoint-05b-tahap-3/qc
  stage2c_deepeval_checkpoint.csv <- /kaggle/input/datasets/muhammadnabilul/checkpoint-05b-tahap-3/qc

isi working dir saat ini:
  environment_snapshot.json  698 bytes
  stage1_prefilter.csv  1610631 bytes
  stage1_summary.json  608 bytes
  stage2_ragas.csv  197219 bytes
  stage2_ragas_checkpoint.csv  88572 bytes
  stage2c_deepeval_checkpoint.csv  380750 bytes


['stage1_prefilter.csv <- /kaggle/input/datasets/muhammadnabilul/checkpoint-05b-tahap-3/qc',
 'stage1_summary.json <- /kaggle/input/datasets/muhammadnabilul/checkpoint-05b-tahap-3/qc',
 'stage2_ragas.csv <- /kaggle/input/datasets/muhammadnabilul/checkpoint-05b-tahap-3/qc',
 'stage2_ragas_checkpoint.csv <- /kaggle/input/datasets/muhammadnabilul/checkpoint-05b-tahap-3/qc',
 'stage2c_deepeval_checkpoint.csv <- /kaggle/input/datasets/muhammadnabilul/checkpoint-05b-tahap-3/qc']

In [6]:
def clean_incomplete_checkpoints(config):
    working_dir = Path(config["output_dir"])
 
    deepeval_active = config.get("deepeval_metrics",
                                 ["faithfulness", "answer_relevancy", "contextual_relevancy"])
    deepeval_cols = [f"deepeval_{m}" for m in deepeval_active]
 
    checkpoint_specs = {
        "stage2_ragas_checkpoint.csv": ["faithfulness", "response_relevancy"],
        "stage2_ragas_checkpoint_test.csv": ["faithfulness", "response_relevancy"],
        "stage2c_deepeval_checkpoint.csv": deepeval_cols,
        "stage2c_deepeval_checkpoint_test.csv": deepeval_cols,
    }
 
    print(f"kriteria lengkap DeepEval: {deepeval_cols}")
 
    for fname, score_cols in checkpoint_specs.items():
        path = working_dir / fname
        if not path.exists() or path.stat().st_size == 0:
            continue
 
        with open(path, encoding="utf-8") as f:
            rows = list(csv.DictReader(f))
        if not rows:
            continue
 
        fieldnames = list(rows[0].keys())
        active_cols = [c for c in score_cols if c in fieldnames]
        if not active_cols:
            print(f"{fname}: kolom skor tidak ditemukan, dilewati")
            continue
 
        keep = [r for r in rows if all(r.get(c, "") not in ("", None, "None") for c in active_cols)]
        dropped = len(rows) - len(keep)
 
        if dropped == 0:
            print(f"{fname}: {len(rows)} record semua lengkap, tidak ada yang dibuang")
            continue
 
        shutil.copy2(path, str(path) + ".bak")
        with open(path, "w", encoding="utf-8", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            w.writerows(keep)
 
        print(f"{fname}: dibuang {dropped} record tidak lengkap, tersisa {len(keep)} valid "
              f"(backup: {fname}.bak)")
 
    print("\nrecord tidak lengkap akan diproses ulang saat stage dijalankan.")
 
 
clean_incomplete_checkpoints(CONFIG)

kriteria lengkap DeepEval: ['deepeval_faithfulness', 'deepeval_answer_relevancy']
stage2_ragas_checkpoint.csv: 2088 record semua lengkap, tidak ada yang dibuang
stage2c_deepeval_checkpoint.csv: dibuang 8 record tidak lengkap, tersisa 884 valid (backup: stage2c_deepeval_checkpoint.csv.bak)

record tidak lengkap akan diproses ulang saat stage dijalankan.


In [7]:
import csv, hashlib

qc = "/kaggle/working/qc"

ck = list(csv.DictReader(open(f"{qc}/stage2_ragas_checkpoint.csv")))
s2 = list(csv.DictReader(open(f"{qc}/stage2_ragas.csv")))
print(f"checkpoint  : {len(ck)} baris")
print(f"stage2_ragas: {len(s2)} baris")

ids_ck = {r["record_id"] for r in ck}
ids_s2 = {r["record_id"] for r in s2}
print(f"identik     : {ids_ck == ids_s2}")
print(f"hanya di ck : {len(ids_ck - ids_s2)}")
print(f"hanya di s2 : {len(ids_s2 - ids_ck)}")

print(f"\nmd5 stage1_prefilter : {hashlib.md5(open(f'{qc}/stage1_prefilter.csv','rb').read()).hexdigest()}")
print(f"md5 stage2_ragas     : {hashlib.md5(open(f'{qc}/stage2_ragas.csv','rb').read()).hexdigest()}")
print(f"md5 checkpoint       : {hashlib.md5(open(f'{qc}/stage2_ragas_checkpoint.csv','rb').read()).hexdigest()}")

from collections import defaultdict
by = defaultdict(list)
for r in s2:
    v = r.get("ragas_faithfulness", "")
    if v not in ("", "None"):
        by[r["subtask_type"]].append(float(v))
print("\nmean per stratum dari stage2_ragas.csv (hasil restore):")
for st in sorted(by):
    print(f"  {st:<28} n={len(by[st]):>4}  mean={sum(by[st])/len(by[st]):.3f}")

checkpoint  : 2088 baris
stage2_ragas: 2088 baris
identik     : True
hanya di ck : 0
hanya di s2 : 0

md5 stage1_prefilter : 2ebd31f0a5e1ee86e080711e5a74b75f
md5 stage2_ragas     : 8947c9b61c48338b3a0ad3520229351e
md5 checkpoint       : 0d80265a226abe72b9773a4329130c84

mean per stratum dari stage2_ragas.csv (hasil restore):
  causal_reasoning             n= 255  mean=0.628
  comparative_analysis         n= 247  mean=0.777
  definition_clarification     n= 209  mean=0.729
  document_overview            n= 156  mean=0.785
  factual_retrieval            n= 269  mean=0.714
  methodology_explanation      n= 222  mean=0.776
  multi_source_synthesis       n= 255  mean=0.695
  source_navigation            n= 222  mean=0.783
  trend_interpretation         n= 253  mean=0.739


### Fungsi umum

In [8]:
def load_records(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records
 
def write_csv(rows, path, fieldnames):
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            writer.writerow(r)
 
def read_csv(path):
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))
 
def write_summary(summary, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
 
def preview_rows(rows, sort_key, n=5, reverse=False, label=""):
    # Gunakan fungsi lambda yang mengecek explicit apakah nilainya None
    ordered = sorted(
        rows, 
        key=lambda r: float(r.get(sort_key)) if r.get(sort_key) is not None else -1.0, 
        reverse=reverse
    )
    print(f"\n--- preview {label} ({'top' if reverse else 'bottom'} {n} by {sort_key}) ---")
    for r in ordered[:n]:
        # Tampilkan data dengan rapi (bisa disesuaikan formatnya jika mau)
        print(f"ID: {r.get('record_id')} | {sort_key}: {r.get(sort_key)} | Subtask: {r.get('subtask_type')}")

## 2. Prefiltering

In [9]:
def compute_indobert_similarity(records, model_name, device, batch_size):
    from sentence_transformers import SentenceTransformer, util
    model = SentenceTransformer(model_name, device=device)
    answers = [r["response"] for r in records]
    contexts = [r.get("context", "") for r in records]
    emb_a = model.encode(answers, convert_to_tensor=True, batch_size=batch_size, show_progress_bar=True)
    emb_c = model.encode(contexts, convert_to_tensor=True, batch_size=batch_size, show_progress_bar=True)
    scores = util.cos_sim(emb_a, emb_c).diagonal().tolist()
    return scores
    
def compute_per_subtask_threshold(rows, stratify_key, global_threshold, z_low=1.0):
    from collections import defaultdict
    by_stratum = defaultdict(list)
    for r in rows:
        by_stratum[r[stratify_key]].append(r["indobert_score"])
 
    thresholds = {}
    for stratum, scores in by_stratum.items():
        n = len(scores)
        if n < 10:
            thresholds[stratum] = global_threshold
            continue
        mean = sum(scores) / n
        var = sum((s - mean) ** 2 for s in scores) / n
        std = var ** 0.5
        adaptive = mean - z_low * std
        thresholds[stratum] = max(adaptive, -1.0)
    return thresholds
 
 
def stage1_prefilter(config):
    records = load_records(config["input_path"])
    scores = compute_indobert_similarity(records, config["embedding_model"], config["device"], config["embed_batch_size"])
 
    raw_rows = []
    for r, s in zip(records, scores):
        raw_rows.append({
            "record_id": r["record_id"],
            "subtask_type": r.get(config["stratify_key"], "?"),
            "generation_focus": r.get("generation_focus", "?"),
            "indobert_score": round(s, 4),
        })
 
    per_stratum_threshold = compute_per_subtask_threshold(
        raw_rows, config["stratify_key"], config["indobert_threshold_flag"], z_low=1.0)
 
    print("\n--- threshold adaptif per subtask_type (mean - 1*std, bukan 0.5 global) ---")
    for stratum, t in sorted(per_stratum_threshold.items()):
        print(f"  {stratum}: threshold={t:.4f}")
 
    rows = []
    for r in raw_rows:
        t = per_stratum_threshold[r["subtask_type"]]
        r["threshold_used"] = round(t, 4)
        r["flag_prefilter"] = r["indobert_score"] < t
        rows.append(r)
 
    write_csv(rows, f"{config['output_dir']}/stage1_prefilter.csv",
              ["record_id", "subtask_type", "generation_focus", "indobert_score", "threshold_used", "flag_prefilter"])
    preview_rows(rows, "indobert_score", n=5, reverse=False, label="stage1 lowest score")
    preview_rows(rows, "indobert_score", n=5, reverse=True, label="stage1 highest score")
    n_flagged = sum(1 for r in rows if r["flag_prefilter"])
    print(f"\nstage1: {n_flagged}/{len(rows)} flagged sebagai prioritas judge ({n_flagged/len(rows)*100:.1f}%)")
    print("[CATATAN] flag_prefilter BUKAN keputusan buang/lolos akhir. "
          "Hanya menentukan siapa yang diprioritaskan masuk sample Ragas/DeepEval di Stage 2.")
    write_summary({
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "total": len(rows),
        "n_flagged": n_flagged,
        "flag_rate": round(n_flagged / len(rows), 4),
        "per_stratum_threshold": {k: round(v, 4) for k, v in per_stratum_threshold.items()},
        "config": {k: config[k] for k in ("embedding_model", "indobert_threshold_flag", "stratify_key")},
    }, f"{config['output_dir']}/stage1_summary.json")
    return rows
    
def run_stage1_if_needed(config):
    path = f"{config['output_dir']}/stage1_prefilter.csv"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        rows = read_csv(path)
        print(f"stage1: SKIP, sudah ada {len(rows)} record di {path}")
        print("        hapus file tersebut jika ingin menghitung ulang embedding.")
        return rows
    print("stage1: file belum ada, menghitung embedding dari awal...")
    if config["device"] == "cpu":
        print("        [PERINGATAN] device=cpu, proses ini akan sangat lama untuk korpus besar.")
    return stage1_prefilter(config)


stage1_rows = run_stage1_if_needed(CONFIG)

stage1: SKIP, sudah ada 20855 record di /kaggle/working/qc/stage1_prefilter.csv
        hapus file tersebut jika ingin menghitung ulang embedding.


## 3. Ragas and deepEval Filtering

In [10]:
def parse_bool(value):
    if isinstance(value, bool):
        return value
    if value is None:
        return False
    s = str(value).strip().lower()
    return s in ("true", "1", "yes")
 
 
def stratified_split_for_ragas(records, prefilter_rows, config):
    flag_map = {r["record_id"]: parse_bool(r["flag_prefilter"]) for r in prefilter_rows}
    by_stratum = defaultdict(lambda: {"flag": [], "general": []})
    for r in records:
        stratum = r.get(config["stratify_key"], "?")
        bucket = "flag" if flag_map.get(r["record_id"], False) else "general"
        by_stratum[stratum][bucket].append(r)
 
    rng = random.Random(config["seed"])
    strata_items = sorted(by_stratum.items())
    if config.get("test_mode") and config.get("max_strata_test"):
        strata_items = strata_items[:config["max_strata_test"]]
        print(f"[TEST MODE] hanya {len(strata_items)} stratum diproses: "
              f"{[s for s, _ in strata_items]}")
 
    sampled = []
    for stratum, groups in strata_items:
        total_stratum = len(groups["flag"]) + len(groups["general"])
        n_target = max(config["min_per_stratum"], math.ceil(total_stratum * config["ragas_sample_rate"]))
        if config["max_per_stratum"]:
            n_target = min(n_target, config["max_per_stratum"])
        n_target = min(n_target, total_stratum)
 
        if config.get("test_mode") and config.get("max_samples_per_stratum_test"):
            n_target = min(n_target, config["max_samples_per_stratum_test"])
 
        n_flag_target = round(n_target * config["flag_ratio"])
        n_flag = min(n_flag_target, len(groups["flag"]))
        n_general = min(n_target - n_flag, len(groups["general"]))
        if n_flag + n_general < n_target:
            remainder = n_target - n_flag - n_general
            remaining_pool = groups["flag"][n_flag:] + groups["general"][n_general:]
            rng.shuffle(remaining_pool)
            extra = remaining_pool[:remainder]
        else:
            extra = []
 
        picked = rng.sample(groups["flag"], n_flag) + rng.sample(groups["general"], n_general) + extra
        print(f"stratum={stratum} total={total_stratum} target={n_target} "
              f"flag={n_flag} general={n_general} extra={len(extra)}")
        sampled.extend(picked)
    return sampled
 
def build_ragas_judge_components(config):
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.metrics import Faithfulness, ResponseRelevancy
    from langchain_openai import ChatOpenAI
    from langchain_huggingface import HuggingFaceEmbeddings
    from kaggle_secrets import UserSecretsClient
 
    api_key = UserSecretsClient().get_secret(config["API_secret_name"])
    model_name=config["ragas_embedding_model"]
    
    print(f"--> Menginisialisasi Ragas Judge menggunakan OpenAI Model: {model_name}")
    
    # Kunci parameter seed di dalam dictionary model_kwargs agar aman dari konflik library
    base_llm = ChatOpenAI(
        model=config["ragas_judge_model"], 
        api_key=api_key, 
        temperature=0, 
        n=1,
        model_kwargs={"seed": config.get("seed", 42)} # Mengunci seed via model_kwargs OpenAI
    )
    llm = LangchainLLMWrapper(base_llm)
    
    emb = HuggingFaceEmbeddings(
        model_name= model_name,
        model_kwargs={"device": config["device"], "trust_remote_code": True},
        encode_kwargs={"batch_size": config["embed_batch_size"]}
    )

    faithfulness_metric = Faithfulness(llm=llm)
    relevancy_metric = ResponseRelevancy(
        llm=llm, embeddings=emb,
        strictness=config.get("ragas_relevancy_strictness", 1))
    
    return llm, emb, faithfulness_metric, relevancy_metric
 
# 1. Gunakan async def 
async def run_ragas_judge(sample, config):
    import asyncio
    from ragas.dataset_schema import SingleTurnSample
 
    llm, emb, faithfulness_metric, relevancy_metric = build_ragas_judge_components(config)
 
    suffix = "_test" if config.get("test_mode") else ""
    checkpoint_path = f"{config['output_dir']}/stage2_ragas_checkpoint{suffix}.csv"
    fieldnames = ["record_id", "faithfulness", "response_relevancy"]
 
    done = {}
    if os.path.exists(checkpoint_path):
        # Gunakan read_csv bawaanmu
        for row in read_csv(checkpoint_path):
            done[row["record_id"]] = row
        print(f"resume: {len(done)} record sudah ada di checkpoint")
    else:
        write_csv([], checkpoint_path, fieldnames)
 
    max_retries = config.get("ragas_max_retries", 5)
    base_delay = config.get("request_delay", 1.0)
 
    for r in sample:
        rid = r["record_id"]
        if rid in done:
            continue
 
        ctx = r.get("context", "")
        if not ctx:
            row = {"record_id": rid, "faithfulness": "", "response_relevancy": ""}
            with open(checkpoint_path, "a", encoding="utf-8", newline="") as f:
                csv.DictWriter(f, fieldnames=fieldnames).writerow(row)
            done[rid] = row
            continue
 
        s = SingleTurnSample(
            user_input=r["instruction"],
            response=r["response"],
            retrieved_contexts=[ctx],
        )
 
        f_val, r_val = None, None
        for attempt in range(max_retries):
            try:
                # 2. Pakai await KARENA KITA SUDAH DI DALAM RUNNING EVENT LOOP
                f_val = await faithfulness_metric.single_turn_ascore(s)
                r_val = await relevancy_metric.single_turn_ascore(s)
                break
            except Exception as e:
                msg = str(e).lower()
                if "429" in msg or "rate" in msg or "timeout" in msg:
                    wait = base_delay * (2 ** attempt)
                    print(f"record_id={rid} attempt={attempt+1} error={e} -> sleep {wait:.1f}s")
                    # 3. GUNAKAN await asyncio.sleep() AGAR TIDAK MEMBLOKIR KERNEL JUPYTER
                    await asyncio.sleep(wait)
                else:
                    print(f"record_id={rid} error non-retryable: {e}")
                    break
 
        row = {
            "record_id": rid,
            "faithfulness": "" if f_val is None else f_val,
            "response_relevancy": "" if r_val is None else r_val,
        }
        with open(checkpoint_path, "a", encoding="utf-8", newline="") as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(row)
            # Memastikan data langsung tertulis di disk storage
            f.flush()
        done[rid] = row

        # Jeda reguler aman antar record (TPM/RPM Guard) menggunakan asinkron sleep
        await asyncio.sleep(base_delay)

    # 4. MEMBACA KEMBALI SELESAI EVALUASI
    scores = []
    for r in sample:
        row = done.get(r["record_id"], {})
        f_val = row.get("faithfulness", "")
        r_val = row.get("response_relevancy", "")
        scores.append({
            "faithfulness": None if f_val == "" else float(f_val),
            "response_relevancy": None if r_val == "" else float(r_val),
        })
    return scores

In [11]:
async def stage2_ragas_judge(config):
    records = load_records(config["input_path"])
    prefilter_rows = read_csv(f"{config['output_dir']}/stage1_prefilter.csv")
    sample = stratified_split_for_ragas(records, prefilter_rows, config)
 
    n_calls = len(sample) * 2
    mode_label = "TEST MODE" if config.get("test_mode") else "FULL RUN"
    print(f"\n[{mode_label}] total sample={len(sample)} -> estimasi API call={n_calls} "
          f"(2 per record: faithfulness + relevancy)")
 
    ragas_scores = await run_ragas_judge(sample, config)
 
    flag_map = {r["record_id"]: r["flag_prefilter"] for r in prefilter_rows}
    indobert_map = {r["record_id"]: r["indobert_score"] for r in prefilter_rows}
    rows = []
    for r, score in zip(sample, ragas_scores):
        rows.append({
            "record_id": r["record_id"],
            "subtask_type": r.get(config["stratify_key"], "?"),
            "generation_focus": r.get("generation_focus", "?"),
            "ragas_faithfulness": score["faithfulness"],
            "ragas_relevancy": score["response_relevancy"],
            "indobert_score": indobert_map.get(r["record_id"]),
            "flag_prefilter": flag_map.get(r["record_id"]),
        })
    suffix = "_test" if config.get("test_mode") else ""
    write_csv(rows, f"{config['output_dir']}/stage2_ragas{suffix}.csv",
              ["record_id", "subtask_type", "generation_focus", "ragas_faithfulness",
               "ragas_relevancy", "indobert_score", "flag_prefilter"])
 
    by_stratum_faith = defaultdict(list)
    by_stratum_relev = defaultdict(list)
    by_focus_faith = defaultdict(list)
    for r in rows:
        if r["ragas_faithfulness"] is not None:
            by_stratum_faith[r["subtask_type"]].append(r["ragas_faithfulness"])
            by_focus_faith[r["generation_focus"]].append(r["ragas_faithfulness"])
        if r["ragas_relevancy"] is not None:
            by_stratum_relev[r["subtask_type"]].append(r["ragas_relevancy"])
 
    for stratum, vals in sorted(by_stratum_faith.items()):
        print(f"stratum={stratum} n={len(vals)} mean_faithfulness={sum(vals)/len(vals):.3f}")
 
    preview_rows(rows, "ragas_faithfulness", n=5, reverse=False, label="stage2 lowest faithfulness")
    preview_rows(rows, "ragas_faithfulness", n=5, reverse=True, label="stage2 highest faithfulness")
    return rows

stage2_rows = await stage2_ragas_judge(CONFIG)

stratum=causal_reasoning total=2549 target=255 flag=128 general=127 extra=0
stratum=comparative_analysis total=2468 target=247 flag=124 general=123 extra=0
stratum=definition_clarification total=2085 target=209 flag=104 general=105 extra=0
stratum=document_overview total=1560 target=156 flag=78 general=78 extra=0
stratum=factual_retrieval total=2690 target=269 flag=134 general=135 extra=0
stratum=methodology_explanation total=2211 target=222 flag=111 general=111 extra=0
stratum=multi_source_synthesis total=2550 target=255 flag=128 general=127 extra=0
stratum=source_navigation total=2215 target=222 flag=111 general=111 extra=0
stratum=trend_interpretation total=2527 target=253 flag=126 general=127 extra=0

[FULL RUN] total sample=2088 -> estimasi API call=4176 (2 per record: faithfulness + relevancy)
--> Menginisialisasi Ragas Judge menggunakan OpenAI Model: nomic-ai/nomic-embed-text-v1.5


/tmp/ipykernel_16/1512457676.py:11: UserWarning: Parameters {'seed'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  ragas_scores = await run_ragas_judge(sample, config)


modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

resume: 2088 record sudah ada di checkpoint
stratum=causal_reasoning n=255 mean_faithfulness=0.628
stratum=comparative_analysis n=247 mean_faithfulness=0.777
stratum=definition_clarification n=209 mean_faithfulness=0.729
stratum=document_overview n=156 mean_faithfulness=0.785
stratum=factual_retrieval n=269 mean_faithfulness=0.714
stratum=methodology_explanation n=222 mean_faithfulness=0.776
stratum=multi_source_synthesis n=255 mean_faithfulness=0.695
stratum=source_navigation n=222 mean_faithfulness=0.783
stratum=trend_interpretation n=253 mean_faithfulness=0.739

--- preview stage2 lowest faithfulness (bottom 5 by ragas_faithfulness) ---
ID: b32c683a29746226 | ragas_faithfulness: 0.0 | Subtask: causal_reasoning
ID: 76f4eab9ccd39acb | ragas_faithfulness: 0.0 | Subtask: causal_reasoning
ID: 1815341543c83900 | ragas_faithfulness: 0.0 | Subtask: causal_reasoning
ID: aa7f8a0f1aa3a721 | ragas_faithfulness: 0.0 | Subtask: causal_reasoning
ID: 45a92f202c140695 | ragas_faithfulness: 0.0 | Sub

In [12]:
def build_deepeval_judge_components(config):
    from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric, ContextualRelevancyMetric
    from deepeval.models import GPTModel
    from kaggle_secrets import UserSecretsClient
    import httpx

    api_key = UserSecretsClient().get_secret(config["API_secret_name"])
    os.environ["OPENAI_API_KEY"] = api_key

    timeout_seconds = config.get("deepeval_request_timeout", 60)
    http_client = httpx.Client(timeout=httpx.Timeout(timeout_seconds, connect=10.0))

    try:
        model = GPTModel(model=config["deepeval_judge_model"],
                         api_key=api_key,
                         temperature=0,
                         http_client=http_client)
        print(f"--> DeepEval Judge: {config['deepeval_judge_model']} "
              f"(http_client timeout={timeout_seconds}s)")
    except TypeError as e:
        model = GPTModel(model=config["deepeval_judge_model"], api_key=api_key, temperature=0)
        print(f"--> DeepEval Judge: {config['deepeval_judge_model']} "
              f"[PERINGATAN] http_client tidak diterima ({e}); timeout internal DeepEval berlaku.")

    registry = {
        "faithfulness": lambda: FaithfulnessMetric(model=model, threshold=0.5, async_mode=False),
        "answer_relevancy": lambda: AnswerRelevancyMetric(model=model, threshold=0.5, async_mode=False),
        "contextual_relevancy": lambda: ContextualRelevancyMetric(model=model, threshold=0.5, async_mode=False),
    }

    enabled = config.get("deepeval_metrics", list(registry.keys()))
    metric_jobs = [(name, registry[name]()) for name in enabled if name in registry]
    print(f"--> Metrik aktif: {[n for n, _ in metric_jobs]}")
    return metric_jobs
 
 
def _gptmodel_accepts_client():
    import inspect
    from deepeval.models import GPTModel
    sig = inspect.signature(GPTModel.__init__)
    return "_openai_client" in sig.parameters
 
 
async def run_deepeval_judge(sample, config):
    import asyncio
    from deepeval.test_case import LLMTestCase
 
    metric_jobs = build_deepeval_judge_components(config)
 
    suffix = "_test" if config.get("test_mode") else ""
    checkpoint_path = f"{config['output_dir']}/stage2c_deepeval_checkpoint{suffix}.csv"
    fieldnames = ["record_id", "deepeval_faithfulness", "deepeval_answer_relevancy",
                  "deepeval_contextual_relevancy", "deepeval_reason_faithfulness",
                  "deepeval_reason_answer_relevancy", "deepeval_reason_contextual_relevancy"]
 
    done = {}
    if os.path.exists(checkpoint_path):
        for row in read_csv(checkpoint_path):
            done[row["record_id"]] = row
        print(f"resume: {len(done)} record sudah ada di checkpoint")
    else:
        write_csv([], checkpoint_path, fieldnames)
 
    max_retries = config.get("deepeval_max_retries", 5)
    base_delay = config.get("deepeval_request_delay", config.get("request_delay", 1.0))
 
    n_processed = 0
    for r in sample:
        rid = r["record_id"]
        if rid in done:
            continue
 
        ctx = r.get("context", "")
        if not ctx:
            row = {k: "" for k in fieldnames}
            row["record_id"] = rid
            with open(checkpoint_path, "a", encoding="utf-8", newline="") as f:
                csv.DictWriter(f, fieldnames=fieldnames).writerow(row)
            done[rid] = row
            continue
 
        test_case = LLMTestCase(
            input=r["instruction"],
            actual_output=r["response"],
            retrieval_context=[ctx],
        )
 
        results = {}
        daily_limit_hit = False
        for metric_name, metric_obj in metric_jobs:
            score, reason = None, ""
            consecutive_ratelimit = 0
            for attempt in range(max_retries):
                try:
                    metric_obj.measure(test_case)
                    score = metric_obj.score
                    reason = metric_obj.reason
                    break
                except Exception as e:
                    msg = str(e).lower()
                    is_ratelimit = "429" in msg or "rate" in msg or "quota" in msg
                    is_timeout = "timeout" in msg
                    if is_ratelimit:
                        consecutive_ratelimit += 1
                    if is_ratelimit or is_timeout:
                        wait = base_delay * (2 ** attempt)
                        print(f"record_id={rid} metric={metric_name} attempt={attempt+1} "
                              f"error={e} -> sleep {wait:.1f}s")
                        await asyncio.sleep(wait)
                    else:
                        print(f"record_id={rid} metric={metric_name} error non-retryable: {e}")
                        break
 
            if score is None and consecutive_ratelimit >= max_retries:
                daily_limit_hit = True
                print(f"\n[STOP] RateLimitError persisten di record_id={rid} metric={metric_name}. "
                      f"Kemungkinan kuota HARIAN (RPD/TPD) habis. Backoff tidak menolong. "
                      f"Progres tersimpan ({n_processed} record baru diproses sesi ini). "
                      f"Lanjutkan besok dengan menjalankan ulang cell ini.")
                break
 
            results[metric_name] = (score, reason)
 
        if daily_limit_hit:
            break
 
        f_val, f_reason = results.get("faithfulness", (None, ""))
        ar_val, ar_reason = results.get("answer_relevancy", (None, ""))
        cr_val, cr_reason = results.get("contextual_relevancy", (None, ""))
 
        row = {
            "record_id": rid,
            "deepeval_faithfulness": "" if f_val is None else f_val,
            "deepeval_answer_relevancy": "" if ar_val is None else ar_val,
            "deepeval_contextual_relevancy": "" if cr_val is None else cr_val,
            "deepeval_reason_faithfulness": (f_reason or "").replace("\n", " "),
            "deepeval_reason_answer_relevancy": (ar_reason or "").replace("\n", " "),
            "deepeval_reason_contextual_relevancy": (cr_reason or "").replace("\n", " "),
        }
        with open(checkpoint_path, "a", encoding="utf-8", newline="") as f:
            csv.DictWriter(f, fieldnames=fieldnames).writerow(row)
            f.flush()
        done[rid] = row
        n_processed += 1
 
        await asyncio.sleep(base_delay)
 
    print(f"\nsesi ini memproses {n_processed} record baru, total di checkpoint: {len(done)}")
 
    scores = []
    for r in sample:
        row = done.get(r["record_id"], {})
        scores.append({
            "deepeval_faithfulness": None if row.get("deepeval_faithfulness", "") == "" else float(row["deepeval_faithfulness"]),
            "deepeval_answer_relevancy": None if row.get("deepeval_answer_relevancy", "") == "" else float(row["deepeval_answer_relevancy"]),
            "deepeval_contextual_relevancy": None if row.get("deepeval_contextual_relevancy", "") == "" else float(row["deepeval_contextual_relevancy"]),
            "deepeval_reason_faithfulness": row.get("deepeval_reason_faithfulness", ""),
            "deepeval_reason_answer_relevancy": row.get("deepeval_reason_answer_relevancy", ""),
            "deepeval_reason_contextual_relevancy": row.get("deepeval_reason_contextual_relevancy", ""),
        })
    return scores

In [13]:
async def stage2c_deepeval_judge(config):
    records = load_records(config["input_path"])
    suffix = "_test" if config.get("test_mode") else ""
    ragas_rows = read_csv(f"{config['output_dir']}/stage2_ragas{suffix}.csv")
    record_lookup = {r["record_id"]: r for r in records}

    sample = []
    for row in ragas_rows:
        rec = record_lookup.get(row["record_id"])
        if rec is None:
            continue
        merged = dict(rec)
        merged["ragas_faithfulness"] = row.get("ragas_faithfulness")
        merged["ragas_relevancy"] = row.get("ragas_relevancy")
        merged["subtask_type"] = row.get("subtask_type", rec.get("subtask_type", "?"))
        sample.append(merged)

    active_metrics = config.get("deepeval_metrics",
                                ["faithfulness", "answer_relevancy", "contextual_relevancy"])
    n_calls = len(sample) * len(active_metrics)
    mode_label = "TEST MODE" if config.get("test_mode") else "FULL RUN"
    print(f"\n[{mode_label}] total sample={len(sample)} -> estimasi API call={n_calls} "
          f"({len(active_metrics)} metrik: {active_metrics})")

    deepeval_scores = await run_deepeval_judge(sample, config)

    rows = []
    for r, score in zip(sample, deepeval_scores):
        rows.append({
            "record_id": r["record_id"],
            "subtask_type": r["subtask_type"],
            "ragas_faithfulness": r.get("ragas_faithfulness"),
            "ragas_relevancy": r.get("ragas_relevancy"),
            "deepeval_faithfulness": score["deepeval_faithfulness"],
            "deepeval_answer_relevancy": score["deepeval_answer_relevancy"],
            "deepeval_contextual_relevancy": score["deepeval_contextual_relevancy"],
            "deepeval_reason_faithfulness": score["deepeval_reason_faithfulness"],
            "deepeval_reason_answer_relevancy": score["deepeval_reason_answer_relevancy"],
            "deepeval_reason_contextual_relevancy": score["deepeval_reason_contextual_relevancy"],
        })

    write_csv(rows, f"{config['output_dir']}/stage2c_deepeval{suffix}.csv",
              ["record_id", "subtask_type", "ragas_faithfulness", "ragas_relevancy",
               "deepeval_faithfulness", "deepeval_answer_relevancy", "deepeval_contextual_relevancy",
               "deepeval_reason_faithfulness", "deepeval_reason_answer_relevancy",
               "deepeval_reason_contextual_relevancy"])

    active_cols = [f"deepeval_{m}" for m in active_metrics]
    by_stratum = defaultdict(lambda: defaultdict(list))
    for r in rows:
        for key in active_cols:
            if r.get(key) is not None:
                by_stratum[r["subtask_type"]][key].append(r[key])

    for stratum, metrics_dict in sorted(by_stratum.items()):
        parts = [f"{k}={sum(v)/len(v):.3f}" for k, v in metrics_dict.items()]
        n_stratum = len(metrics_dict[active_cols[0]]) if active_cols[0] in metrics_dict else 0
        print(f"stratum={stratum} n={n_stratum} " + " ".join(parts))

    preview_rows(rows, "deepeval_faithfulness", n=5, reverse=False,
                 label="stage2c lowest deepeval faithfulness")
    if "contextual_relevancy" in active_metrics:
        preview_rows(rows, "deepeval_contextual_relevancy", n=5, reverse=False,
                     label="stage2c lowest contextual relevancy")
    return rows


stage2c_rows = await stage2c_deepeval_judge(CONFIG)


[FULL RUN] total sample=2088 -> estimasi API call=4176 (2 metrik: ['faithfulness', 'answer_relevancy'])


Output()

--> DeepEval Judge: gpt-4o-mini (http_client timeout=60s)
--> Metrik aktif: ['faithfulness', 'answer_relevancy']
resume: 884 record sudah ada di checkpoint


record_id=87696f2aab79df3f metric=faithfulness attempt=1 error=RetryError[<Future at 0x7ff5d11dc950 state=finished raised TimeoutError>] -> sleep 1.0s


Output()

...

Output()

Output()

record_id=6ca02bf11be1f595 metric=answer_relevancy attempt=1 error=Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} -> sleep 1.0s


Output()

record_id=6ca02bf11be1f595 metric=answer_relevancy attempt=2 error=Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}} -> sleep 2.0s

[STOP] RateLimitError persisten di record_id=6ca02bf11be1f595 metric=answer_relevancy. Kemungkinan kuota HARIAN (RPD/TPD) habis. Backoff tidak menolong. Progres tersimpan (767 record baru diproses sesi ini). Lanjutkan besok dengan menjalankan ulang cell ini.

sesi ini memproses 767 record baru, total di checkpoint: 1651
stratum=causal_reasoning n=254 deepeval_faithfulness=0.859 deepeval_answer_relevancy=0.937
stratum=comparative_analysis n=246 deepeval_faithfulness=0.814 deepeval_answer_relevancy=0.963
stratum=definition_clarification n=206 deepeval_faithfulness=0.886 deepeval_answer_relevancy=0.922


In [14]:
def build_deepeval_output_from_checkpoint(config):
    suffix = "_test" if config.get("test_mode") else ""
    ck_path = f"{config['output_dir']}/stage2c_deepeval_checkpoint{suffix}.csv"
    out_path = f"{config['output_dir']}/stage2c_deepeval{suffix}.csv"

    if not os.path.exists(ck_path) or os.path.getsize(ck_path) == 0:
        print("checkpoint DeepEval belum ada, tidak ada yang bisa ditulis.")
        return None

    ck = {r["record_id"]: r for r in read_csv(ck_path)}
    ragas_rows = read_csv(f"{config['output_dir']}/stage2_ragas{suffix}.csv")

    active_metrics = config.get("deepeval_metrics",
                                ["faithfulness", "answer_relevancy", "contextual_relevancy"])
    required_cols = [f"deepeval_{m}" for m in active_metrics]

    fieldnames = ["record_id", "subtask_type", "ragas_faithfulness", "ragas_relevancy",
                  "deepeval_faithfulness", "deepeval_answer_relevancy",
                  "deepeval_contextual_relevancy", "deepeval_reason_faithfulness",
                  "deepeval_reason_answer_relevancy", "deepeval_reason_contextual_relevancy"]

    rows, n_incomplete = [], 0
    for r in ragas_rows:
        rid = r["record_id"]
        c = ck.get(rid)
        if not c:
            continue
        if any(c.get(col, "") in ("", "None") for col in required_cols):
            n_incomplete += 1
            continue
        row = {k: "" for k in fieldnames}
        row["record_id"] = rid
        row["subtask_type"] = r.get("subtask_type", "?")
        row["ragas_faithfulness"] = r.get("ragas_faithfulness", "")
        row["ragas_relevancy"] = r.get("ragas_relevancy", "")
        for k in fieldnames[4:]:
            row[k] = c.get(k, "")
        rows.append(row)

    write_csv(rows, out_path, fieldnames)

    n_ragas = len(ragas_rows)
    print(f"stage2c_deepeval{suffix}.csv ditulis ke: {out_path}")
    print(f"  metrik wajib lengkap : {required_cols}")
    print(f"  record valid         : {len(rows):,}")
    print(f"  tercatat tapi tidak lengkap : {n_incomplete:,}")
    print(f"  cakupan sampel Ragas : {len(rows):,}/{n_ragas:,} "
          f"({len(rows)/n_ragas*100:.1f}%)")

    if len(rows) < n_ragas:
        print(f"\n[CATATAN] DeepEval belum mencakup seluruh sampel Ragas. "
              f"Analisis perbandingan Ragas-DeepEval berlaku untuk subset n={len(rows):,}; "
              f"laporkan angka ini apa adanya sebagai keterbatasan teknis (latensi API).")

    by = defaultdict(lambda: defaultdict(list))
    for r in rows:
        for col in required_cols:
            v = r.get(col, "")
            if v not in ("", "None"):
                by[r["subtask_type"]][col].append(float(v))

    if by:
        print("\nrata-rata per stratum:")
        for st in sorted(by):
            parts = [f"{c.replace('deepeval_',''):<20}={sum(v)/len(v):.3f}"
                     for c, v in by[st].items()]
            n_st = len(by[st][required_cols[0]])
            print(f"  {st:<28} n={n_st:>4}  " + "  ".join(parts))

    return rows


deepeval_output = build_deepeval_output_from_checkpoint(CONFIG)

stage2c_deepeval.csv ditulis ke: /kaggle/working/qc/stage2c_deepeval.csv
  metrik wajib lengkap : ['deepeval_faithfulness', 'deepeval_answer_relevancy']
  record valid         : 1,605
  tercatat tapi tidak lengkap : 46
  cakupan sampel Ragas : 1,605/2,088 (76.9%)

[CATATAN] DeepEval belum mencakup seluruh sampel Ragas. Analisis perbandingan Ragas-DeepEval berlaku untuk subset n=1,605; laporkan angka ini apa adanya sebagai keterbatasan teknis (latensi API).

rata-rata per stratum:
  causal_reasoning             n= 254  faithfulness        =0.859  answer_relevancy    =0.937
  comparative_analysis         n= 246  faithfulness        =0.814  answer_relevancy    =0.963
  definition_clarification     n= 206  faithfulness        =0.886  answer_relevancy    =0.921
  document_overview            n= 156  faithfulness        =0.872  answer_relevancy    =0.958
  factual_retrieval            n= 243  faithfulness        =0.843  answer_relevancy    =0.950
  methodology_explanation      n= 217  faithf

## 4. Selected human eval

In [15]:
def select_human_eval_sample(ragas_rows, config):
    scored = [r for r in ragas_rows if r["ragas_faithfulness"] not in (None, "")]
    for r in scored:
        r["ragas_faithfulness"] = float(r["ragas_faithfulness"])
 
    if not scored:
        print("[WARNING] tidak ada record dengan ragas_faithfulness terisi, human eval sample kosong")
        return []
 
    eval_n = config["human_eval_n_test"] if config.get("test_mode") else config["human_eval_n"]
    eval_n = min(eval_n, len(scored))
 
    scores = sorted(r["ragas_faithfulness"] for r in scored)
    mid = scores[len(scores) // 2]
    for r in scored:
        r["_dist_to_median"] = abs(r["ragas_faithfulness"] - mid)
 
    n_border = round(eval_n * config["human_borderline_frac"])
    n_anchor = eval_n - n_border
 
    by_stratum = defaultdict(list)
    for r in scored:
        by_stratum[r["subtask_type"]].append(r)
 
    rng = random.Random(config["seed"])
    selected = []
    n_strata = len(by_stratum)
    for stratum, rows_ in sorted(by_stratum.items()):
        border_quota = max(1, round(n_border / n_strata))
        anchor_quota = max(1, round(n_anchor / n_strata))
        ordered_border = sorted(rows_, key=lambda r: r["_dist_to_median"])[:border_quota]
        remaining = [r for r in rows_ if r not in ordered_border]
        remaining_sorted = sorted(remaining, key=lambda r: r["ragas_faithfulness"])
        anchor_pool = remaining_sorted[:anchor_quota // 2] + remaining_sorted[-(anchor_quota - anchor_quota // 2):]
        selected.extend(ordered_border + anchor_pool)
 
    selected = selected[:eval_n]
    mode_label = "TEST MODE" if config.get("test_mode") else "FULL RUN"
    print(f"[{mode_label}] human eval sample: {len(selected)} records (target={eval_n})")
    by_stratum_count = defaultdict(int)
    for r in selected:
        by_stratum_count[r["subtask_type"]] += 1
    for stratum, cnt in sorted(by_stratum_count.items()):
        print(f"stratum={stratum} n_selected={cnt}")
    return selected

In [16]:
def stage3_human_eval_template(config):
    records = load_records(config["input_path"])
    suffix = "_test" if config.get("test_mode") else ""
    ragas_rows = read_csv(f"{config['output_dir']}/stage2_ragas{suffix}.csv")
    record_lookup = {r["record_id"]: r for r in records}
    selected = select_human_eval_sample(ragas_rows, config)
    rows = []
    for r in selected:
        rec = record_lookup.get(r["record_id"], {})
        rows.append({
            "record_id": r["record_id"],
            "subtask_type": r["subtask_type"],
            "generation_focus": r.get("generation_focus", "?"),
            "question": rec.get("instruction", ""),
            "answer": rec.get("response", ""),
            "context": rec.get("context", ""),
            "ragas_faithfulness": r["ragas_faithfulness"],
            "indobert_score": r["indobert_score"],
            "human_label": "",
        })
    write_csv(rows, f"{config['output_dir']}/stage3_human_eval_template{suffix}.csv",
              ["record_id", "subtask_type", "generation_focus", "question", "answer",
               "context", "ragas_faithfulness", "indobert_score", "human_label"])
    print(f"stage3: {len(rows)} baris siap diisi manual")
 
    by_stratum_count = defaultdict(int)
    for r in selected:
        by_stratum_count[r["subtask_type"]] += 1
 
    write_summary({
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
        "n_selected": len(rows),
        "per_subtask": dict(sorted(by_stratum_count.items())),
        "config": {k: config[k] for k in ("human_eval_n", "human_eval_n_test", "human_borderline_frac", "seed", "test_mode")},
    }, f"{config['output_dir']}/stage3_summary{suffix}.json")
    return rows
 
 
stage3_rows = stage3_human_eval_template(CONFIG)

[FULL RUN] human eval sample: 99 records (target=100)
stratum=causal_reasoning n_selected=11
stratum=comparative_analysis n_selected=11
stratum=definition_clarification n_selected=11
stratum=document_overview n_selected=11
stratum=factual_retrieval n_selected=11
stratum=methodology_explanation n_selected=11
stratum=multi_source_synthesis n_selected=11
stratum=source_navigation n_selected=11
stratum=trend_interpretation n_selected=11
stage3: 99 baris siap diisi manual
